# Inputs for Neural Net
- Type of game played
- Length of game (time and moves)
- Game outcomes
    - Winner
    - Type of outcome
- Move evaluation
    - Variance
    - Mean
    - IQR
    - Next best moves
        - Mean
        - Did they choose the best move?
        - Did they choose the next best move?
- Piece evaluation
    - Number of moves of each piece
    - Captures per piece
    - Number of checks
    - Checks per piece
    - Breakdown per part of the game

# Environment Notes
To get the project to run properly, the python version needs to be set to 3.11. Create a fresh conda envioronment. Run the following commands:
```bash
conda create -n chess python-3.11
conda activate chess
pip install pandas
pip install matplotlib
pip install seaborn
pip install chess
pip install tensorflow
```

In [5]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import chess

# Loading and Cleaning

In [6]:
# Import data and observe shape and columns
chess_data_raw = pd.read_csv("data/games.csv")
print(chess_data_raw.shape)
print(chess_data_raw.columns)
chess_data_raw.head()

chess_data_raw = chess_data_raw[chess_data_raw['turns'] > 4]

chess_data_raw

chess_data = chess_data_raw.drop(columns=["id", "white_id", "black_id"])
chess_data = chess_data.dropna()

# Drop all non-rated games for doing rating analysis
chess_data = chess_data[chess_data["rated"] == True]
# Remove rated, created at, and last move at columns
chess_data = chess_data.drop(["rated","created_at","last_move_at"],axis=1)
# Map winner column
chess_data["winner"] = chess_data["winner"].map({"black": -1, "white": 1}).fillna(0)
# Map victory column
chess_data["victory_status"] = chess_data["victory_status"].map({"resign": 0, "mate": 1,"draw": 2, "outoftime": 3})
# Convert moves column to array of moves
chess_data["moves"] = chess_data["moves"].str.split(" ")
# Add column for average player ELO per game
chess_data["game_rating"] = (chess_data["white_rating"] + chess_data["black_rating"]) / 2
# Add column to bucket games by rating 
chess_data["rating_bucket"] = pd.cut(
    chess_data["game_rating"],
    bins=3,
    labels=["low", "medium", "high"]
)

print(chess_data.shape)
chess_data.head()

(20058, 16)
Index(['id', 'rated', 'created_at', 'last_move_at', 'turns', 'victory_status',
       'winner', 'increment_code', 'white_id', 'white_rating', 'black_id',
       'black_rating', 'moves', 'opening_eco', 'opening_name', 'opening_ply'],
      dtype='str')
(15907, 12)


,turns,victory_status,winner,increment_code,white_rating,black_rating,moves,opening_eco,opening_name,opening_ply,game_rating,rating_bucket
1,16,0,-1.0,5+10,1322,1261,"[d4, Nc6, e4, e5, f4, f6, dxe5, fxe5, fxe5, Nx...",B00,Nimzowitsch Defense: Kennedy Variation,4,1291.5,low
2,61,1,1.0,5+10,1496,1500,"[e4, e5, d3, d6, Be3, c6, Be2, b5, Nd2, a5, a4...",C20,King's Pawn Game: Leonardis Variation,3,1498.0,medium
3,61,1,1.0,20+0,1439,1454,"[d4, d5, Nf3, Bf5, Nc3, Nf6, Bf4, Ng4, e3, Nc6...",D02,Queen's Pawn Game: Zukertort Variation,3,1446.5,medium
4,95,1,1.0,30+3,1523,1469,"[e4, e5, Nf3, d6, d4, Nc6, d5, Nb4, a3, Na6, N...",C41,Philidor Defense,5,1496.0,medium
6,33,0,1.0,10+0,1520,1423,"[d4, d5, e4, dxe4, Nc3, Nf6, f3, exf3, Nxf3, N...",D00,Blackmar-Diemer Gambit: Pietrowsky Defense,10,1471.5,medium


In [7]:
# -------------------------------------------------------
# Use Python Chess to get parameters for neural net
# -------------------------------------------------------


def board_analysis(movelist):
    piece_lookup = {
        chess.PAWN: "pawn",
        chess.KNIGHT: "knight",
        chess.BISHOP: "bishop",
        chess.ROOK: "rook",
        chess.QUEEN: "queen",
        chess.KING: "king",
    }
    colors = ["white", "black"]
    categories = ["moves", "captures", "checks", "forks", "pins"]
    pieces = ["pawn", "knight", "bishop", "rook", "queen", "king"]

    analysis = {
        color: {
            category: {piece: 0 for piece in pieces}
            for category in categories
        }
        for color in colors
    }
    promotion_counts = {color: 0 for color in colors}

    board = chess.Board()

    for san in movelist:
        try:
            move = board.parse_san(san)
        except Exception:
            break

        color = "white" if board.turn == chess.WHITE else "black"
        piece = board.piece_at(move.from_square)

        if piece is None:
            break

        piece_name = piece_lookup[piece.piece_type]
        analysis[color]["moves"][piece_name] += 1

        if board.is_capture(move):
            analysis[color]["captures"][piece_name] += 1

        board.push(move)

        if board.is_check():
            analysis[color]["checks"][piece_name] += 1

        attacked_enemy_pieces = 0
        pinned_enemy_pieces = 0
        for square in board.attacks(move.to_square):
            target_piece = board.piece_at(square)
            if target_piece is not None and target_piece.color != piece.color:
                attacked_enemy_pieces += 1
                if board.is_pinned(target_piece.color, square):
                    pinned_enemy_pieces += 1

        if attacked_enemy_pieces >= 2:
            analysis[color]["forks"][piece_name] += 1

        if pinned_enemy_pieces > 0:
            analysis[color]["pins"][piece_name] += pinned_enemy_pieces

        if move.promotion is not None:
            promotion_counts[color] += 1

    flattened_vector = []
    for color in colors:
        for category in categories:
            for piece in pieces:
                flattened_vector.append(analysis[color][category][piece])

    for color in colors:
        flattened_vector.append(promotion_counts[color])

    return flattened_vector


In [13]:
colors = ["white", "black"]
categories = ["moves", "captures", "checks", "forks", "pins"]
pieces = ["pawn", "knight", "bishop", "rook", "queen", "king"]

neural_net_columns = [
    f"{color}_{category}_{piece}"
    for color in colors
    for category in categories
    for piece in pieces
] + [f"{color}_promotions" for color in colors]

neural_net_columns += ["white_elo", "black_elo"]

neural_net_rows = []

for idx, row in chess_data.iterrows():
    board_analysis_row = board_analysis(row["moves"])
    board_analysis_row += [row["white_rating"], row["black_rating"]]
    neural_net_rows.append(board_analysis_row)

neural_net_input = pd.DataFrame(neural_net_rows, columns=neural_net_columns)
neural_net_input.describe()


,white_moves_pawn,white_moves_knight,white_moves_bishop,white_moves_rook,white_moves_queen,white_moves_king,white_captures_pawn,white_captures_knight,white_captures_bishop,white_captures_rook,...,black_pins_pawn,black_pins_knight,black_pins_bishop,black_pins_rook,black_pins_queen,black_pins_king,white_promotions,black_promotions,white_elo,black_elo
count,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,...,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000,15907.000000
mean,8.623310,5.590118,4.977180,4.452820,4.019991,4.041177,1.845414,1.367071,1.433331,1.175457,...,0.041868,0.060477,0.307412,0.242032,0.404665,0.003520,0.086629,0.080405,1600.194317,1596.472811
std,4.373822,3.432520,3.043119,5.254442,3.748805,5.680511,1.373962,1.241637,1.194892,1.491020,...,0.212479,0.255435,0.595403,0.620097,0.742038,0.073446,0.328134,0.314789,282.669026,287.344443
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,784.000000,789.000000
25%,5.000000,3.000000,3.000000,1.000000,2.000000,1.000000,1.000000,0.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1399.000000,1394.000000
50%,8.000000,5.000000,4.000000,3.000000,3.000000,2.000000,2.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1579.000000,1576.000000
75%,11.000000,7.000000,6.000000,6.000000,5.000000,5.000000,3.000000,2.000000,2.000000,2.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1796.000000,1792.000000
max,35.000000,59.000000,44.000000,83.000000,41.000000,74.000000,8.000000,11.000000,11.000000,9.000000,...,3.000000,4.000000,5.000000,8.000000,11.000000,4.000000,5.000000,5.000000,2622.000000,2588.000000


# Nueral Network Development and Design
- Experiment with multiple types of models
- Use smaller data to test
- Find which performs the best
- Train the best performing model using large data

## Import Tensorflow and Relevant Libraries
- Using tensorflow because it is flexible and fast
- It will allow us to process the relatively large dataset (13k rows)

In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Model 1
- Inputs:
    - Action counters (checks, forks, pins, moves, and promotions per piece per color)
    - No centipawn data

- Question: Can ELO be predicted without stockfish centipawn calculation and purely based off of board state and piece statistics?

In [20]:
# ==============================
# Seperate data into X and y
# ==============================

feature_cols = [col for col in neural_net_input.columns if col not in ['white_elo', 'black_elo']]
target_cols = ['white_elo', 'black_elo']

X = neural_net_input[feature_cols].values.astype(np.float32)
y = neural_net_input[target_cols].values.astype(np.float32)

# ============================================================
# Split data into test, train, and validation
# ============================================================

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=42
)

# ===================================
# Create normalization layer (scaling)
# ===================================

normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train)

# ==============================
# Build model
# ==============================
model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    normalizer,
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(2)
])


model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=[keras.metrics.MeanAbsoluteError()]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )
]

# ==============================
# Train
# ==============================
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# ==============================
# Evaluate
# ==============================
pred = model.predict(X_test)

white_mae = mean_absolute_error(y_test[:, 0], pred[:, 0])
black_mae = mean_absolute_error(y_test[:, 1], pred[:, 1])

white_rmse = np.sqrt(mean_squared_error(y_test[:, 0], pred[:, 0]))
black_rmse = np.sqrt(mean_squared_error(y_test[:, 1], pred[:, 1]))

print(f"White ELO MAE:  {white_mae:.2f}")
print(f"Black ELO MAE:  {black_mae:.2f}")
print(f"White ELO RMSE: {white_rmse:.2f}")
print(f"Black ELO RMSE: {black_rmse:.2f}")

Epoch 1/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 626us/step - loss: 906290.1875 - mean_absolute_error: 749.6625 - val_loss: 153940.5469 - val_mean_absolute_error: 308.4721
Epoch 2/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 446us/step - loss: 127908.8750 - mean_absolute_error: 283.5698 - val_loss: 114781.2500 - val_mean_absolute_error: 266.3078
Epoch 3/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 448us/step - loss: 105765.2422 - mean_absolute_error: 258.7498 - val_loss: 102712.5000 - val_mean_absolute_error: 252.5441
Epoch 4/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 448us/step - loss: 95481.1250 - mean_absolute_error: 246.0424 - val_loss: 94013.3750 - val_mean_absolute_error: 241.9861
Epoch 5/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 447us/step - loss: 88795.7109 - mean_absolute_error: 237.3436 - val_loss: 87788.1953 - val_mean_absolute_error: 234.1609
Epoch 6/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 447us/step - loss: 83615.4844 - mean_absolute_error: 230.3090 - val_loss: 85081.8750 - val_mean_absolute_error: 230.0671
Epoc

# Analysis
This model performed quite poorly. The mean error was over 200, which is not ideal for ELO. ELO ranges from around 100 - ~3400. The middle 50% of players in this dataset are within 1400 - 1800. This means that our error is too large to accurately guess ELO.

Conclusion: ELO cannot be accurately guessed purely by the statistics calculated for the model. ...